In [ ]:
!pip install monai itk einops nibabel torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 55.4 MB/s eta 0:00:00


In [ ]:
import os
import torch
import monai.transforms as mt
import monai
from monai.networks.nets import SwinUNETR
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete
from torch.utils.data import DataLoader, random_split
from pathlib import Path
from tqdm import tqdm
from google.colab import drive

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [ ]:
drive.mount('/content/drive', force_remount=True)

IMAGES_DIR = "/content/drive/MyDrive/panther/ImagesTr"
LABELS_DIR = "/content/drive/MyDrive/panther/LabelsTr"
IMAGE_UNLABELED_DIR = "/content/drive/MyDrive/panther/ImagesTr_unlabeled"
PSEUDO_LABELS_DIR = "/content/drive/MyDrive/pseudo_labels_SwinUNETR"
PSEUDO_WEIGHTS_DIR = "/content/drive/MyDrive/pseudo_weights_SwinUNETR"
OUTPUT_DIR = "/content/drive/MyDrive/SwinUNETR_models/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
class SegmentationDataSet(monai.data.Dataset):
    def __init__(self, imagesTr, labelsTr):
        images = sorted(Path(imagesTr).glob("*.mha"))
        labels = sorted(Path(labelsTr).glob("*.mha"))
        data = [{"image": str(img), "label": str(lbl)} for img, lbl in zip(images, labels)]

        transforms = mt.Compose([
            mt.LoadImaged(keys=["image", "label"], reader="ITKReader"),
            mt.EnsureChannelFirstd(keys=["image", "label"]),
            mt.Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=["bilinear", "nearest"]),
            mt.Orientationd(keys=["image", "label"], axcodes="RAS"),
            mt.NormalizeIntensityd(keys=["image"]),
            mt.RandCropByPosNegLabeld(
                keys=["image", "label"], label_key="label",
                spatial_size=(96, 96, 96), pos=3, neg=1, num_samples=4,
            ),
            mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=0),
            mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=1),
            mt.RandFlipd(keys=["image", "label"], prob=0.2, spatial_axis=2),
            mt.RandRotate90d(keys=["image", "label"], prob=0.2, max_k=3),
            mt.RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.2),
            mt.RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.2),
        ])
        super().__init__(data=data, transform=transforms)

In [ ]:
class PseudoDataSet(monai.data.Dataset):
    def __init__(self, imagesTr_unlabeled, pseudo_labels, pseudo_weights):
        self.image_paths_unlabeled = sorted(Path(imagesTr_unlabeled).glob("*.mha"))
        self.pseudo_paths_labels = sorted(Path(pseudo_labels).glob("*.mha"))
        self.pseudo_paths_weights = sorted(Path(pseudo_weights).glob("*.mha"))
        data = []
        for path_label, path_weight in zip(self.pseudo_paths_labels, self.pseudo_paths_weights):
            image_name = path_label.name.replace("_pseudo", "")
            image_path = Path(imagesTr_unlabeled) / image_name
            if image_path.exists():
                image_path = Path(imagesTr_unlabeled) / path_label.name.replace("_pseudo", "")
                data.append({
                    "image": str(image_path),
                    "label": str(path_label),
                    "weight": str(path_weight)
                })

        transforms = mt.Compose([
            mt.LoadImaged(
                keys=["image", "label", "weight"],
                reader="ITKReader"
            ),
            mt.EnsureChannelFirstd(keys=["image", "label", "weight"]),
            mt.Spacingd(
                keys=["image", "label", "weight"],
                pixdim=(1.0, 1.0, 1.0),
                mode=["bilinear", "nearest", "nearest"]
            ),
            mt.Orientationd(
                keys=["image", "label", "weight"],
                axcodes="RAS"
            ),
            mt.NormalizeIntensityd(keys=["image"]),
            mt.RandCropByPosNegLabeld(
                keys=["image", "label", "weight"],
                label_key="label",
                spatial_size=(96, 96, 96),
                pos=3,
                neg=1,
                num_samples=2,
            ),
            mt.RandFlipd(
                keys=["image", "label", "weight"],
                prob=0.2,
                spatial_axis=0
            ),
            mt.RandFlipd(
                keys=["image", "label", "weight"],
                prob=0.2,
                spatial_axis=1
            ),
            mt.RandFlipd(
                keys=["image", "label", "weight"],
                prob=0.2, spatial_axis=2)
            ,
            mt.RandRotate90d(
                keys=["image", "label", "weight"],
                prob=0.2,
                max_k=3
            ),
            mt.RandScaleIntensityd(
                keys=["image"],
                factors=0.1,
                prob=0.2
            ),
            mt.RandShiftIntensityd(
                keys=["image"],
                offsets=0.1,
                prob=0.2
            ),
            mt.ResizeWithPadOrCropd(
                keys=["image", "label", "weight"],
                spatial_size=(96, 96, 96)
            ),
        ])

        super().__init__(data=data, transform=transforms)

In [ ]:
def train(epochs, lr):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    dataset_label = SegmentationDataSet(IMAGES_DIR, LABELS_DIR)
    dataset_pseudo = PseudoDataSet(IMAGE_UNLABELED_DIR, PSEUDO_LABELS_DIR, PSEUDO_WEIGHTS_DIR)
    print(f"Pseudo dataset size: {len(dataset_pseudo)}")
    torch.manual_seed(42)
    train_size = int(0.85 * len(dataset_label))
    val_size = len(dataset_label) - train_size
    labeled_train, val_dataset = random_split(dataset_label, [train_size, val_size])

    train_loader_labeled = DataLoader(labeled_train, batch_size=4, shuffle=True, num_workers=1)
    train_loader_pseudo = DataLoader(dataset_pseudo, batch_size=4, shuffle=True, num_workers=3)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

    model = SwinUNETR(in_channels=1, out_channels=3, feature_size=48, spatial_dims=3).to(device)
    model.load_state_dict(torch.load(
        "/content/drive/MyDrive/SwinUNETR_models/SwinUNETR_random_phase2_v2.pth",
        map_location=device
    ))
    print("Loaded weights")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)
    loss_fn = DiceLoss(to_onehot_y=True, softmax=True, weight=torch.tensor([0.02, 0.85, 0.13]).to(device))
    post_pred = AsDiscrete(argmax=True, to_onehot=3)
    post_label = AsDiscrete(to_onehot=3)
    dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
    best_dice = 0.0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for batch in tqdm(train_loader_labeled, desc=f"Epoch {epoch+1}/{epochs} [labeled]"):
            for sample in batch:
                image = sample["image"].to(device)
                label = sample["label"].to(device)
                optimizer.zero_grad()
                output = model(image)
                loss = loss_fn(output, label)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

        for batch in tqdm(train_loader_pseudo, desc=f"Epoch {epoch+1}/{epochs} [pseudo]"):
            for sample in batch:
                image = sample["image"].to(device)
                label = sample["label"].to(device)
                weight = sample["weight"].to(device)
                optimizer.zero_grad()
                output = model(image)
                loss = loss_fn(output, label) * weight.mean()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

        avg_loss = epoch_loss / (len(train_loader_labeled) + len(train_loader_pseudo))

        model.eval()
        with torch.no_grad():
            for batch in val_loader:
                for sample in batch:
                    image = sample["image"].to(device)
                    label = sample["label"].to(device)
                    output = model(image)
                    output_post = post_pred(output[0])
                    label_post = post_label(label[0])
                    dice_metric(y_pred=output_post.unsqueeze(0), y=label_post.unsqueeze(0))

        dice_per_class = dice_metric.aggregate()
        dice_pancreas = dice_per_class[1].item()
        dice_tumor = dice_per_class[0].item()
        dice_metric.reset()
        scheduler.step()

        print(f"Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f} — Dice pancreatic: {dice_pancreas:.4f} — Dice tumor: {dice_tumor:.4f}")

        if dice_tumor > best_dice:
            best_dice = dice_tumor
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "SwinUNETR_phase4_07_v3.pth"))
            print(f"Model saved (Dice tumor: {best_dice:.4f})")

In [ ]:
train(epochs=150, lr=1e-4)

Using device: cuda


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


Pseudo dataset size: 166
Loaded weights


Epoch 1/150 [pseudo]: 100%|██████████| 42/42 [06:37<00:00,  9.46s/it]


Epoch 1/150 — Loss: 0.2576 — Dice pancreatic: 0.5869 — Dice tumor: 0.2665
Model saved (Dice tumor: 0.2665)


...

Epoch 148/150 — Loss: 0.2384 — Dice pancreatic: 0.6725 — Dice tumor: 0.4253


Epoch 149/150 [pseudo]: 100%|██████████| 42/42 [01:24<00:00,  2.01s/it]


Epoch 149/150 — Loss: 0.2449 — Dice pancreatic: 0.6697 — Dice tumor: 0.4817


Epoch 150/150 [pseudo]: 100%|██████████| 42/42 [01:24<00:00,  2.02s/it]


Epoch 150/150 — Loss: 0.2236 — Dice pancreatic: 0.6595 — Dice tumor: 0.4509
